# 노트북 03 — 래칫팅과 키 노출

대칭 래칫을 10개의 메시지 동안 굴려보고, 5단계에서 키 노출을 시뮬레이션하여 무엇이 안전하고 무엇이 그렇지 않은지 확인합니다.

In [ ]:
from pqmsg.kdf import derive_chain_step
from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305
import os

## 단계 1 — 체인이 전진하는 모습 관찰

In [ ]:
chain_key = os.urandom(32)
for i in range(10):
    msg_key, next_ck = derive_chain_step(chain_key)
    aead = ChaCha20Poly1305(msg_key)
    nonce = os.urandom(12)
    ct = aead.encrypt(nonce, f"message {i}".encode(), None)
    print(f"msg {i}: msg_key[:4]={msg_key[:4].hex()}  chain_key[:4]={next_ck[:4].hex()}")
    chain_key = next_ck

## 단계 2 — 5단계에서 노출 시뮬레이션

공격자가 5단계 시점(메시지 4가 전송된 후)의 `chain_key`를 훔쳤다고 가정합니다.

In [ ]:
chain_key = os.urandom(32)
stolen = None
saved_keys = []
for i in range(10):
    if i == 5:
        stolen = chain_key
    msg_key, next_ck = derive_chain_step(chain_key)
    saved_keys.append(msg_key)
    chain_key = next_ck

print("stolen chain_key (first 8B):", stolen[:8].hex())

## 단계 3 — 공격자가 과거 메시지를 복호화할 수 있을까?

훔친 chain_key로부터 공격자는 메시지 키 0..4를 유도하려고 시도합니다.

In [ ]:
ck = stolen
attacker_keys_future = []
for i in range(5, 10):
    mk, nxt = derive_chain_step(ck)
    attacker_keys_future.append(mk)
    ck = nxt

print("Future messages (5..9): attacker reconstructs every msg_key.")
for i in range(5, 10):
    matches = attacker_keys_future[i - 5] == saved_keys[i]
    print(f"  msg {i} key reconstructed: {matches}")

print("\nPast messages (0..4): attacker has chain_key at step 5, and HKDF is one-way.")
print("  There is NO function that turns next_chain into previous chain_key.")
print("  Attacker CANNOT compute msg_keys 0..4 from `stolen`.")

## 핵심 정리

1. **전방 비밀성은 유지됩니다**: 노출 이전의 메시지는 HKDF가 단방향이기 때문에 기밀이 유지됩니다.
2. **후방 비밀성은 깨집니다**: 노출 시점 이후의 모든 메시지는 읽을 수 있게 됩니다.

**완전한 Double Ratchet**은 몇 메시지마다 DH 단계를 추가합니다 — 새로운 X25519/ML-KEM 교환이 공격자가 보지 못한 엔트로피로 체인 키를 갱신합니다. 이것이 후방 비밀성 공백을 메웁니다. 여기서는 그 한계를 명확하게 드러내기 위해 의도적으로 생략했습니다. 실제 메신저(Signal, iMessage PQ3)에서는 항상 완전한 래칫을 사용해야 합니다.